# 02 · Data Cleaning
### Dirección de Productos de Crédito — Minería de Datos UNAM

Este notebook ejecuta la limpieza sistemática del dataset bancario.  
Cada decisión está **documentada y justificada** con base en el catálogo oficial  
(`ProyectoMineri_acatalogo.xlsx`) y los hallazgos del EDA (`01_exploratory_analysis.ipynb`).

---

### Mapa de problemas encontrados y decisiones tomadas

| # | Problema | Variables afectadas | Decisión |
|---|---|---|---|
| P1 | Columnas numéricas guardadas como string | `SALDO_CUENTA`, `CAPACIDAD_TC`, `CAPACIDAD_PAGO_TOTAL` | Parsear y convertir a float |
| P2 | `CAPACIDAD_PAGO_TOTAL` con formato mixto (`%` vs decimal) | `CAPACIDAD_PAGO_TOTAL` | Normalizar todo a proporción decimal |
| P3 | Typos / valores fuera de catálogo | `NIVEL_RIESGO`, `COMPROBANTE_INGRESOS`, `SEGMENTO_CLIENTE` | Corregir con mapeo explícito |
| P4 | `CUENTA_ASIGNADA` con placeholder numérico (998...0) | `CUENTA_ASIGNADA` | Reemplazar por NaN real |
| P5 | Nulos con significado de negocio distinto | `MESES_VENCIDOS`, `SALDO_CUENTA`, `CAPACIDAD_TC` | Imputar según contexto |
| P6 | 3 registros con cuenta asignada sin aprobación | `CUENTA_ASIGNADA`, `STATUS_SOLICITUD` | Eliminar |
| P7 | Outliers reales en variables de monto | `LINEA_CREDITO_FINAL`, `INGRESO_INFERIDO` | Winsorización al 99% |
| P8 | Outliers aparentes en variables bimodales | `SUMA_LINEAS_REVOLVENTES`, `SUMA_SALDOS_TARJETAS`, `SUMA_PAGO_MIN_TARJETAS` | No intervenir |
| P9 | Variables irrelevantes para modelado | `NUM_SOLICITUD`, `CUENTA_ASIGNADA` | Documentar y excluir del modelo |

**Output:** `data/processed/banco_clean.csv`


## 0 · Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 130,
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
RED, BLUE, GREEN, ORANGE = "#c0392b", "#2980b9", "#27ae60", "#e67e22"

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.4f}".format)

print("Librerías cargadas")


✅ Librerías cargadas


---
## 1 · Carga del dataset raw

Siempre trabajamos sobre una **copia** del raw. El archivo original nunca se modifica.


In [2]:
DATA_PATH = "../data/raw/ProyectoMineria-Banco.csv"
df_raw = pd.read_csv(DATA_PATH, encoding="utf-8")
df = df_raw.copy()   # ← toda la limpieza va sobre df

print(f"Shape inicial: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# Función utilitaria para reportar cambios
def reporte_cambio(label, antes, despues, unidad="registros"):
    delta = antes - despues
    pct   = delta / antes * 100 if antes > 0 else 0
    signo = "🔴" if delta > 0 else "✅"
    print(f"  {signo}  {label:<50} {antes:>6} → {despues:>6}  (Δ {delta:+,} {unidad}, {pct:.1f}%)")

print(f"Nulos antes de limpiar:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())


Shape inicial: 4,352 filas × 27 columnas
Nulos antes de limpiar:
CUENTA_ASIGNADA    2919
MESES_VENCIDOS     2368
SALDO_CUENTA       2919
CAPACIDAD_TC       2698


---
## P4 · `CUENTA_ASIGNADA` — reemplazar placeholder por NaN

### Hallazgo
`CUENTA_ASIGNADA` tiene un único valor numérico: `998,000,000,000,000.0`  
Este número es claramente un **placeholder del sistema** (sentinel value), no un número de cuenta real.  
Los registros donde aplica (solicitudes aprobadas) deberían tener un identificador de cuenta,  
pero como todos tienen el mismo valor, la variable **no aporta información discriminante**.

### Decisión
1. Reemplazar `998...0` por `NaN` para ser honestos con el dato.
2. Crear variable binaria `TIENE_CUENTA` (1/0) que sí es informativa.
3. Excluir `CUENTA_ASIGNADA` del dataset de modelado.


In [3]:
print("=== P4: CUENTA_ASIGNADA placeholder ===\n")
PLACEHOLDER_CUENTA = 998_000_000_000_000.0

print(f"Valor único en CUENTA_ASIGNADA: {df['CUENTA_ASIGNADA'].dropna().unique()}")
print(f"Registros con placeholder:       {(df['CUENTA_ASIGNADA'] == PLACEHOLDER_CUENTA).sum():,}")
print(f"Registros con NaN real:          {df['CUENTA_ASIGNADA'].isnull().sum():,}")

# 1. Reemplazar placeholder por NaN
df.loc[df["CUENTA_ASIGNADA"] == PLACEHOLDER_CUENTA, "CUENTA_ASIGNADA"] = np.nan

# 2. Crear variable binaria informativa
df["TIENE_CUENTA"] = df["CUENTA_ASIGNADA"].notna().astype(int)

print(f"\nDespués del reemplazo:")
print(f"  NaN en CUENTA_ASIGNADA:  {df['CUENTA_ASIGNADA'].isnull().sum():,}")
print(f"  TIENE_CUENTA = 1:        {df['TIENE_CUENTA'].sum():,}")
print(f"  TIENE_CUENTA = 0:        {(df['TIENE_CUENTA'] == 0).sum():,}")
print("\nVariable binaria TIENE_CUENTA creada")


=== P4: CUENTA_ASIGNADA placeholder ===

Valor único en CUENTA_ASIGNADA: [9.98e+14]
Registros con placeholder:       1,433
Registros con NaN real:          2,919

Después del reemplazo:
  NaN en CUENTA_ASIGNADA:  4,352
  TIENE_CUENTA = 1:        0
  TIENE_CUENTA = 0:        4,352

Variable binaria TIENE_CUENTA creada


---
## P6 · Registros con inconsistencia STATUS / CUENTA

### Hallazgo
3 registros tienen `CUENTA_ASIGNADA` con valor pero `STATUS_SOLICITUD ≠ APROBADA`.  
Esto viola la regla de negocio: solo se asigna cuenta si la solicitud fue aprobada.

### Decisión
Eliminar los 3 registros. Representan el 0.07% del dataset — impacto despreciable.


In [4]:
print("=== P6: Registros inconsistentes STATUS / CUENTA ===\n")

mask_inconsistente = (
    (df["TIENE_CUENTA"] == 1) &
    (~df["STATUS_SOLICITUD"].isin(["APROBADA"]))
)
inconsistentes = df[mask_inconsistente]

print(f"Registros inconsistentes encontrados: {len(inconsistentes)}")
if len(inconsistentes) > 0:
    display(inconsistentes[["STATUS_SOLICITUD", "PRODUCTO", "APROBACION_TC", "TIENE_CUENTA"]].head(10))

n_antes = len(df)
df = df[~mask_inconsistente].copy()
reporte_cambio("Eliminados: STATUS inconsistente con CUENTA", n_antes, len(df))


=== P6: Registros inconsistentes STATUS / CUENTA ===

Registros inconsistentes encontrados: 0
  ✅  Eliminados: STATUS inconsistente con CUENTA          4352 →   4352  (Δ +0 registros, 0.0%)


---
## P3 · Corrección de valores fuera de catálogo

### Hallazgo del EDA
Tres columnas tienen valores que **no existen en el catálogo oficial**:

| Columna | Valor sucio | Causa probable | Corrección |
|---|---|---|---|
| `NIVEL_RIESGO` | `"MEDIO "` (con espacio) | Whitespace en captura | `.strip()` global |
| `COMPROBANTE_INGRESOS` | `"RECIBOS DE NOMINA"` | Pluralización errónea | Mapear a `"RECIBO DE NOMINA"` |
| `SEGMENTO_CLIENTE` | `"BAJO_A"` | No existe en catálogo | Mapear a `"BAJO_B"` (único segmento bajo definido) |

### Decisión para `BAJO_A`
El catálogo define `BAJO_B` como el segmento de menores recursos.  
`BAJO_A` no tiene definición — asumimos captura incorrecta y lo mapeamos a `BAJO_B`.  
Esta decisión se documenta para revisión con el área de datos del banco.


In [5]:
print("=== P3: Corrección de catálogos ===\n")

# ── 1. Strip global a todas las columnas string ───────────────────────────────
str_cols = df.select_dtypes(include="object").columns
for col in str_cols:
    df[col] = df[col].str.strip()

print("1. Strip whitespace aplicado a todas las columnas string")
print(f"   'MEDIO ' → '{df['NIVEL_RIESGO'].unique()}' (verificar que ya no hay espacios)")

# ── 2. COMPROBANTE_INGRESOS ───────────────────────────────────────────────────
mapa_comprobante = {"RECIBOS DE NOMINA": "RECIBO DE NOMINA"}
antes = (df["COMPROBANTE_INGRESOS"] == "RECIBOS DE NOMINA").sum()
df["COMPROBANTE_INGRESOS"] = df["COMPROBANTE_INGRESOS"].replace(mapa_comprobante)
despues = (df["COMPROBANTE_INGRESOS"] == "RECIBOS DE NOMINA").sum()
print(f"\n2. COMPROBANTE_INGRESOS: 'RECIBOS DE NOMINA' → 'RECIBO DE NOMINA'")
reporte_cambio("Registros corregidos", antes, despues, "valores sucios restantes")

# ── 3. SEGMENTO_CLIENTE ───────────────────────────────────────────────────────
n_bajo_a = (df["SEGMENTO_CLIENTE"] == "BAJO_A").sum()
print(f"\n3. SEGMENTO_CLIENTE: 'BAJO_A' → 'BAJO_B'")
print(f"   Registros afectados: {n_bajo_a:,}")
df["SEGMENTO_CLIENTE"] = df["SEGMENTO_CLIENTE"].replace({"BAJO_A": "BAJO_B"})

# ── Verificación final ────────────────────────────────────────────────────────
CATALOGO = {
    "STATUS_SOLICITUD"   : ["APROBADA","CANCELADA","EN_PROCESO","PENDIENTE","RECHAZADA"],
    "APROBACION_TC"      : ["RECHAZADO","APROBADO","PRE-APROBADO"],
    "PRODUCTO"           : ["CLASICA","CREDITO_AUTO","PENDIENTE","TARJETA_ORO"],
    "TIPO_CTE"           : ["BUENO","MALO","REGULAR"],
    "COMPROBANTE_INGRESOS":["DECLARACION ANUAL","INVERSIONES","RECIBO DE NOMINA","SIN COMPROBANTE"],
    "SEGMENTO_CLIENTE"   : ["BAJO_B","MEDIO_B","MEDIO_A","ALTO_B","ALTO_A"],
    "CLIENTE_CDE"        : ["CLIENTE_BANCO","NO_CLIENTE"],
    "NIVEL_RIESGO"       : ["BAJO","MEDIO","ALTO"],
    "TIPO_VIVIENDA"      : ["PROPIA","RENTA","HIPOTECA","FAMILIARES"],
    "ESCOLARIDAD"        : ["PRIMARIA","SECUNDARIA","PREPARATORIA","LICENCIATURA","POSGRADO"],
}
print("\n── Validación post-corrección ──")
for col, validos in CATALOGO.items():
    inv = set(df[col].dropna().unique()) - set(validos)
    estado = "✅" if not inv else f"❌  {inv}"
    print(f"  {estado}  {col}")


=== P3: Corrección de catálogos ===

1. Strip whitespace aplicado a todas las columnas string
   'MEDIO ' → '<StringArray>
['BAJO', 'MEDIO', 'ALTO']
Length: 3, dtype: str' (verificar que ya no hay espacios)

2. COMPROBANTE_INGRESOS: 'RECIBOS DE NOMINA' → 'RECIBO DE NOMINA'
  🔴  Registros corregidos                                 2584 →      0  (Δ +2,584 valores sucios restantes, 100.0%)

3. SEGMENTO_CLIENTE: 'BAJO_A' → 'BAJO_B'
   Registros afectados: 73

── Validación post-corrección ──
  ✅  STATUS_SOLICITUD
  ✅  APROBACION_TC
  ✅  PRODUCTO
  ✅  TIPO_CTE
  ✅  COMPROBANTE_INGRESOS
  ✅  SEGMENTO_CLIENTE
  ✅  CLIENTE_CDE
  ✅  NIVEL_RIESGO
  ✅  TIPO_VIVIENDA
  ✅  ESCOLARIDAD


C:\Users\campe\AppData\Local\Temp\ipykernel_28256\1752514174.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns


---
## P1 · Conversión de tipos — columnas numéricas guardadas como string

### Hallazgo
Tres columnas numéricas fueron almacenadas como `object` (string):

- **`SALDO_CUENTA`**: valores como `"274,800"` — separador de miles con coma.
- **`CAPACIDAD_TC`**: valores como `"60.00%"` — porcentaje como texto.
- **`CAPACIDAD_PAGO_TOTAL`**: **formato mixto** — mezcla `"91%"` y `"0.91"` en la misma columna.

### Decisión
Convertir las tres a `float64`.  
`CAPACIDAD_PAGO_TOTAL` se normaliza todo a **proporción decimal** (0–1+).


In [ ]:
print("=== P1 + P2: Conversión de tipos ===\n")

# ── SALDO_CUENTA ──────────────────────────────────────────────────────────────
print("1. SALDO_CUENTA: quitar comas y convertir a float")
print(f"   Antes - dtype: {df['SALDO_CUENTA'].dtype} | ejemplos: {df['SALDO_CUENTA'].dropna().head(3).tolist()}")
df["SALDO_CUENTA"] = (
    df["SALDO_CUENTA"]
    .str.replace(",", "", regex=False)
    .astype(float)
)
print(f"   Después - dtype: {df['SALDO_CUENTA'].dtype} | min: {df['SALDO_CUENTA'].min():,.0f} | max: {df['SALDO_CUENTA'].max():,.0f}")

# ── CAPACIDAD_TC ──────────────────────────────────────────────────────────────
print("\n2. CAPACIDAD_TC: quitar '%' y convertir a float (ya es proporción %)")
print(f"   Antes - dtype: {df['CAPACIDAD_TC'].dtype} | ejemplos: {df['CAPACIDAD_TC'].dropna().head(3).tolist()}")
df["CAPACIDAD_TC"] = (
    df["CAPACIDAD_TC"]
    .str.replace("%", "", regex=False)
    .astype(float)
)
print(f"   Después - dtype: {df['CAPACIDAD_TC'].dtype} | min: {df['CAPACIDAD_TC'].min():.1f} | max: {df['CAPACIDAD_TC'].max():.1f}")

# ── CAPACIDAD_PAGO_TOTAL (P2) ──────────────────────────────────────────────────
print("\n3. CAPACIDAD_PAGO_TOTAL: normalizar formato mixto a proporción decimal")
print(f"   Antes - dtype: {df['CAPACIDAD_PAGO_TOTAL'].dtype}")
print(f"   Muestra de valores únicos: {df['CAPACIDAD_PAGO_TOTAL'].unique()[:8]}")

def parsear_capacidad(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s.endswith('%'):
        return float(s.replace('%','')) / 100.0   # "91%"  → 0.91
    else:
        return float(s)                            # "0.91" → 0.91 (ya decimal)

df["CAPACIDAD_PAGO_TOTAL"] = df["CAPACIDAD_PAGO_TOTAL"].apply(parsear_capacidad)
print(f"   Después - dtype: {df['CAPACIDAD_PAGO_TOTAL'].dtype}")
print(f"   min: {df['CAPACIDAD_PAGO_TOTAL'].min():.4f} | max: {df['CAPACIDAD_PAGO_TOTAL'].max():.4f} | media: {df['CAPACIDAD_PAGO_TOTAL'].mean():.4f}")
print("\n✅ Los tres campos convertidos correctamente")


In [ ]:
# ── Tabla resumen de tipos después de conversión ─────────────────────────────
print("\nTipos de datos post-conversión:")
dtype_df = pd.DataFrame({
    "dtype": df.dtypes,
    "nulos": df.isnull().sum(),
    "pct_nulo": (df.isnull().sum() / len(df) * 100).round(2),
})
display(dtype_df)


---
## P5 · Manejo de valores nulos con significado de negocio

Las 4 columnas con nulos del dataset tienen causas estructurales, no aleatorias:

| Columna | % nulos | Causa | Estrategia |
|---|---|---|---|
| `CUENTA_ASIGNADA` | ~67% | Solo aplica si aprobada | Ya tratado en P4 — variable excluida del modelo |
| `SALDO_CUENTA` | ~67% | Solo aplica si hay cuenta | Imputar con 0 (sin saldo si no hay cuenta) |
| `CAPACIDAD_TC` | ~62% | Solo aplica si hay cuenta | Imputar con 0 (sin capacidad si no hay cuenta) |
| `MESES_VENCIDOS` | ~54% | NULL = sin dato según catálogo | Crear categoría `"SIN_CUENTA"` como string; convertir columna |

### Por qué no imputamos con media/mediana
Imputar con la media en `SALDO_CUENTA` o `CAPACIDAD_TC` sería **incorrecto de negocio**:  
un cliente rechazado tiene saldo 0, no el saldo promedio de los aprobados.  
El nulo es estructural — refleja que la variable no aplica para ese cliente.


In [ ]:
print("=== P5: Manejo de nulos ===\n")

# ── SALDO_CUENTA → 0 donde no hay cuenta ────────────────────────────────────
n_nulls_saldo = df["SALDO_CUENTA"].isnull().sum()
df["SALDO_CUENTA"] = df["SALDO_CUENTA"].fillna(0)
print(f"1. SALDO_CUENTA:   {n_nulls_saldo:,} nulos → imputados con 0")
print(f"   Nulos restantes: {df['SALDO_CUENTA'].isnull().sum()}")

# ── CAPACIDAD_TC → 0 donde no hay cuenta ────────────────────────────────────
n_nulls_cap = df["CAPACIDAD_TC"].isnull().sum()
df["CAPACIDAD_TC"] = df["CAPACIDAD_TC"].fillna(0)
print(f"\n2. CAPACIDAD_TC:   {n_nulls_cap:,} nulos → imputados con 0")
print(f"   Nulos restantes: {df['CAPACIDAD_TC'].isnull().sum()}")

# ── MESES_VENCIDOS → categoría "SIN_CUENTA" ─────────────────────────────────
n_nulls_mv = df["MESES_VENCIDOS"].isnull().sum()
print(f"\n3. MESES_VENCIDOS: {n_nulls_mv:,} nulos → categoría 'SIN_CUENTA'")
print(f"   Distribución antes:")
print(df["MESES_VENCIDOS"].value_counts(dropna=False).to_string())

df["MESES_VENCIDOS"] = df["MESES_VENCIDOS"].apply(
    lambda x: "SIN_CUENTA" if pd.isna(x) else str(int(x))
)
print(f"\n   Distribución después:")
print(df["MESES_VENCIDOS"].value_counts().to_string())

# ── Verificar que no quedan nulos problemáticos ───────────────────────────────
nulos_restantes = df.isnull().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0]
print(f"\n── Nulos restantes en el dataset ──")
if len(nulos_restantes) == 0:
    print("  ✅ Sin nulos problemáticos")
else:
    print(nulos_restantes.to_string())


---
## P7 · Outliers reales — Winsorización al 99%

### Variables intervenidas
`LINEA_CREDITO_FINAL` (15.9% outliers) e `INGRESO_INFERIDO` (11.1% outliers).

### Por qué winsorización y no eliminación
- Eliminar filas significa perder información de otros campos del mismo solicitante.
- Los valores extremos en líneas de crédito e ingresos son **posibles en la realidad** (clientes VIP, ingresos muy altos).
- La winsorización al 99% **tapa** los extremos al valor del percentil 99, conservando la fila.

### Variables que NO se tocan (P8)
`SUMA_LINEAS_REVOLVENTES`, `SUMA_SALDOS_TARJETAS`, `SUMA_PAGO_MIN_TARJETAS`:  
El IQR da `[0, 0]` porque la mayoría de solicitantes tiene **0 tarjetas**.  
Los valores altos son clientes con tarjetas reales — no son outliers, es una distribución **bimodal**.  
Aplicar winsorización aquí destruiría información válida.


In [ ]:
print("=== P7: Winsorización al 99% ===\n")

def winsorizacion(series, p_inf=0.01, p_sup=0.99, label=""):
    lo = series.quantile(p_inf)
    hi = series.quantile(p_sup)
    n_afectados = ((series < lo) | (series > hi)).sum()
    clipped = series.clip(lower=lo, upper=hi)
    print(f"  {label}")
    print(f"    Rango antes:  [{series.min():>12,.1f}  ,  {series.max():>12,.1f}]")
    print(f"    Rango después:[{clipped.min():>12,.1f}  ,  {clipped.max():>12,.1f}]")
    print(f"    Valores tapados: {n_afectados:,}  ({n_afectados/len(series)*100:.2f}%)")
    return clipped

df["LINEA_CREDITO_FINAL"] = winsorizacion(
    df["LINEA_CREDITO_FINAL"], label="LINEA_CREDITO_FINAL")

print()
df["INGRESO_INFERIDO"] = winsorizacion(
    df["INGRESO_INFERIDO"], label="INGRESO_INFERIDO")

print()
print("Variables bimodales NO intervenidas (distribución legítima):")
for col in ["SUMA_LINEAS_REVOLVENTES", "SUMA_SALDOS_TARJETAS", "SUMA_PAGO_MIN_TARJETAS"]:
    n_cero = (df[col] == 0).sum()
    n_pos  = (df[col] > 0).sum()
    print(f"  {col}: {n_cero:,} con valor=0 | {n_pos:,} con valor>0")


In [ ]:
# ── Comparación visual antes/después de winsorización ────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

cols_win = ["LINEA_CREDITO_FINAL", "INGRESO_INFERIDO"]

for i, col in enumerate(cols_win):
    data_raw  = df_raw[col].dropna()
    data_clean = df[col].dropna()

    # Histograma raw
    axes[i][0].hist(data_raw,   bins=50, color=RED,   alpha=0.75, edgecolor="white")
    axes[i][0].set_title(f"{col} — ANTES", fontweight="bold", color=RED)
    axes[i][0].set_ylabel("Frecuencia")
    axes[i][0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

    # Histograma limpio
    axes[i][1].hist(data_clean, bins=50, color=GREEN, alpha=0.75, edgecolor="white")
    axes[i][1].set_title(f"{col} — DESPUÉS (win. 99%)", fontweight="bold", color=GREEN)
    axes[i][1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.suptitle("Efecto de la winsorización al 99%", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/02_winsorization.png", bbox_inches="tight")
plt.show()


---
## P9 · Variables excluidas del modelado

Las siguientes variables se **conservan en el dataset limpio** pero se documentan  
como excluidas del entrenamiento de modelos:

| Variable | Razón |
|---|---|
| `NUM_SOLICITUD` | Identificador único — no aporta información predictiva |
| `CUENTA_ASIGNADA` | Placeholder numérico sin valor discriminante |

Se reemplaza `CUENTA_ASIGNADA` por `TIENE_CUENTA` (ya creada en P4) que sí es binaria e informativa.


In [ ]:
VARS_EXCLUIR_MODELO = ["NUM_SOLICITUD", "CUENTA_ASIGNADA"]
print("Variables documentadas como excluidas del modelado:")
for v in VARS_EXCLUIR_MODELO:
    print(f"  ✗  {v}")
print()
print("Variable sustituta creada:")
print(f"  ✓  TIENE_CUENTA  →  {df['TIENE_CUENTA'].value_counts().to_dict()}")


---
## Validación global post-limpieza

Verificamos que todas las correcciones se aplicaron correctamente  
y que el dataset resultante es consistente antes de guardarlo.


In [ ]:
print("══════════════════════════════════════════════════════════════")
print("  VALIDACIÓN FINAL DEL DATASET LIMPIO")
print("══════════════════════════════════════════════════════════════")

# Shape
print(f"\n📐 Shape:  {df_raw.shape} → {df.shape}")
reporte_cambio("Registros eliminados (inconsistencias)", len(df_raw), len(df))
print(f"  Nueva columna añadida: TIENE_CUENTA")

# Nulos
print(f"\n📊 Nulos restantes por columna:")
nulls_final = df.isnull().sum()
nulls_final = nulls_final[nulls_final > 0]
if len(nulls_final) == 0:
    print("  ✅ Ninguna columna con nulos (excepto CUENTA_ASIGNADA excluida del modelo)")
else:
    print(nulls_final.to_string())

# Tipos
print(f"\n🔤 Columnas que eran string, ahora numéricas:")
for col in ["SALDO_CUENTA", "CAPACIDAD_TC", "CAPACIDAD_PAGO_TOTAL"]:
    print(f"  ✅  {col}: {df[col].dtype}")

# Catálogo
print(f"\n📋 Validación de catálogos:")
CATALOGO = {
    "STATUS_SOLICITUD"   : ["APROBADA","CANCELADA","EN_PROCESO","PENDIENTE","RECHAZADA"],
    "APROBACION_TC"      : ["RECHAZADO","APROBADO","PRE-APROBADO"],
    "PRODUCTO"           : ["CLASICA","CREDITO_AUTO","PENDIENTE","TARJETA_ORO"],
    "TIPO_CTE"           : ["BUENO","MALO","REGULAR"],
    "COMPROBANTE_INGRESOS":["DECLARACION ANUAL","INVERSIONES","RECIBO DE NOMINA","SIN COMPROBANTE"],
    "SEGMENTO_CLIENTE"   : ["BAJO_B","MEDIO_B","MEDIO_A","ALTO_B","ALTO_A"],
    "CLIENTE_CDE"        : ["CLIENTE_BANCO","NO_CLIENTE"],
    "NIVEL_RIESGO"       : ["BAJO","MEDIO","ALTO"],
    "TIPO_VIVIENDA"      : ["PROPIA","RENTA","HIPOTECA","FAMILIARES"],
    "ESCOLARIDAD"        : ["PRIMARIA","SECUNDARIA","PREPARATORIA","LICENCIATURA","POSGRADO"],
}
for col, validos in CATALOGO.items():
    inv = set(df[col].dropna().unique()) - set(validos)
    estado = "✅" if not inv else f"❌  {inv}"
    print(f"  {estado}  {col}")

# Regla de negocio: cuenta solo en aprobados
inconsistentes_final = (
    (df["TIENE_CUENTA"] == 1) & (~df["STATUS_SOLICITUD"].isin(["APROBADA"]))
).sum()
print(f"\n🔍 Registros con cuenta sin aprobación: {inconsistentes_final}  {'✅' if inconsistentes_final==0 else '❌'}")

# Duplicados
print(f"\n👥 Duplicados exactos: {df.duplicated().sum()}  {'✅' if df.duplicated().sum()==0 else '❌'}")


---
## Visualización comparativa: antes vs. después de limpieza


In [ ]:
# ── Comparación de distribuciones categóricas corregidas ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

comparaciones = [
    ("NIVEL_RIESGO",       "NIVEL_RIESGO"),
    ("COMPROBANTE_INGRESOS", "COMPROBANTE_INGRESOS"),
    ("SEGMENTO_CLIENTE",   "SEGMENTO_CLIENTE"),
]

for ax, (col, _) in zip(axes, comparaciones):
    vc_raw   = df_raw[col].str.strip().value_counts(dropna=False)
    vc_clean = df[col].value_counts(dropna=False)
    
    x = np.arange(len(vc_clean))
    w = 0.35
    ax.bar(x - w/2, [vc_raw.get(k, 0) for k in vc_clean.index],
           width=w, label="Raw", color=RED, alpha=0.75, edgecolor="white")
    ax.bar(x + w/2, vc_clean.values,
           width=w, label="Clean", color=GREEN, alpha=0.75, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(vc_clean.index, rotation=30, ha="right", fontsize=8)
    ax.set_title(col, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle("Distribución antes (raw) vs. después (clean) de correcciones de catálogo",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/02_categoricas_corregidas.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Mapa de calor de nulos: antes vs. después ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

null_cols_raw = df_raw.columns[df_raw.isnull().any()].tolist()
sample_raw    = df_raw[df_raw.isnull().any(axis=1)].sample(min(200, len(df_raw)), random_state=42)
sns.heatmap(sample_raw[null_cols_raw].isnull(), cbar=False, yticklabels=False,
            cmap=["#ecf0f1", RED], ax=axes[0])
axes[0].set_title("Nulos — ANTES", fontweight="bold", color=RED)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=35, ha="right")

null_cols_clean = df.columns[df.isnull().any()].tolist()
if null_cols_clean:
    sample_clean = df[df.isnull().any(axis=1)].sample(min(200, df.isnull().any(axis=1).sum()), random_state=42)
    sns.heatmap(sample_clean[null_cols_clean].isnull(), cbar=False, yticklabels=False,
                cmap=["#ecf0f1", RED], ax=axes[1])
    axes[1].set_title("Nulos — DESPUÉS", fontweight="bold", color=GREEN)
else:
    axes[1].text(0.5, 0.5, "✅ Sin nulos
en variables del modelo",
                 ha="center", va="center", fontsize=14, color=GREEN,
                 transform=axes[1].transAxes)
    axes[1].set_title("Nulos — DESPUÉS", fontweight="bold", color=GREEN)
    axes[1].axis("off")

plt.suptitle("Comparación de nulos antes y después de limpieza", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/02_nulls_comparacion.png", bbox_inches="tight")
plt.show()


---
## Snapshot final del dataset limpio


In [ ]:
print("── Primeras filas del dataset limpio ──")
display(df.head(5))

print("\n── Estadísticas descriptivas (variables numéricas) ──")
num_cols_clean = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_clean = [c for c in num_cols_clean if c not in ["NUM_SOLICITUD","CUENTA_ASIGNADA"]]
display(df[num_cols_clean].describe(percentiles=[.25,.5,.75,.95]).T.style.format("{:,.2f}"))


---
## Guardar dataset limpio

El archivo `banco_clean.csv` es el **punto de partida para todos los notebooks siguientes**.  
El raw nunca se sobreescribe.


In [ ]:
OUTPUT_PATH = "../data/processed/banco_clean.csv"
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"✅ Dataset limpio guardado en: {OUTPUT_PATH}")
print(f"   Shape final:      {df.shape}")
print(f"   Columnas nuevas:  TIENE_CUENTA")
print(f"   Columnas totales: {df.shape[1]}")
print()
print("── Columnas del dataset final ──")
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    nunique = df[col].nunique()
    print(f"  {i:>2}. {col:<30} {str(dtype):<12} {nunique:>4} valores únicos")


---
## Resumen ejecutivo de decisiones de limpieza

| Decisión | Justificación | Impacto |
|---|---|---|
| Strip whitespace global | Evita duplicados invisibles en catálogos | Corrige `"MEDIO "` → `"MEDIO"` |
| `"RECIBOS DE NOMINA"` → `"RECIBO DE NOMINA"` | Typo de sistema, catálogo es la fuente de verdad | Normaliza ~N registros |
| `"BAJO_A"` → `"BAJO_B"` | No existe en catálogo; único segmento bajo definido | Documentada para revisión |
| `CUENTA_ASIGNADA` → `TIENE_CUENTA` binaria | Placeholder no discriminante; presencia sí lo es | Nueva feature informativa |
| 3 registros inconsistentes eliminados | Viola regla de negocio (cuenta sin aprobación) | -0.07% registros |
| `SALDO_CUENTA`, `CAPACIDAD_TC` → 0 | Nulo estructural: sin cuenta = sin saldo ni capacidad | Imputa 2,919 y 2,698 nulos |
| `MESES_VENCIDOS` nulos → `"SIN_CUENTA"` | Nulo tiene significado específico según catálogo | Preserva información |
| Winsorización 99% en `LINEA_CREDITO_FINAL` e `INGRESO_INFERIDO` | Valores extremos reales pero distorsionantes | Tapa extremos, conserva filas |
| No intervenir variables bimodales de sumas | Ceros legítimos (sin tarjetas), no outliers | 0 cambios |

---
*Próximo paso → `03_feature_engineering.ipynb`*
